# Time series and forecasting

*Trend, seasonality, smoothing, and forecast evaluation*

The warehouse managers at Prairie Wholesale plan staffing and stock one quarter ahead. If the revenue of each of the next 13 weeks were known, the managers could schedule more temporary staff and order more stock for the weeks before a peak, and fewer for the slow weeks. The revenue of future weeks is not known, but it can be estimated from the revenue of past weeks. Such an estimate is called a **forecast**. In forecasting, the past values of a time series are used to predict the future values of the same series. The figure shows the idea with hypothetical data.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as _np
import plotly.graph_objects as _go

_rng = _np.random.default_rng(3)
_weeks = _np.arange(104 + 26)
_pattern = 100 + 0.15 * _weeks + 25 * _np.exp(-((_weeks % 52 - 20) / 3.5) ** 2)
_past = _pattern[:104] + _rng.normal(0, 6, 104)
_fig = _go.Figure()
_fig.add_trace(_go.Scatter(x=_weeks[:104], y=_past, mode="lines", name="Past revenue (known)",
                           line=dict(color="#4c78a8", width=2)))
_fig.add_trace(_go.Scatter(x=_weeks[103:], y=_np.r_[_past[-1], _pattern[104:]], mode="lines", name="Forecast",
                           line=dict(color="#f58518", width=3, dash="dash")))
_fig.add_vline(x=103, line=dict(color="#9aa1a9", width=1.5, dash="dot"))
_fig.add_annotation(x=103, y=1.02, yref="paper", text="Today", showarrow=False, yanchor="bottom")
_fig.add_annotation(x=124, y=_pattern[124] + 4, text="Next peak", showarrow=True, arrowhead=2,
                    ax=-50, ay=-30)
_fig.update_layout(height=340, margin=dict(t=40), legend=dict(orientation="h", y=-0.2),
                   xaxis_title="Week", yaxis_title="Weekly revenue (thousands of dollars)")
_fig

In this chapter, we build such a forecast for the weekly revenue of Prairie Wholesale, 13 weeks ahead.

As in Chapters 9 and 10, we score every method on a test set, that is, on data that was not used to fit the model. For a time series, however, the test set is formed differently. In Chapter 9, the customers were assigned to the training set and the test set at random, which was possible because the order of the customers did not matter. The weeks of a time series are in a fixed order, and a forecast is always made for weeks that have not yet happened. The held-out weeks must therefore be later than every week used for fitting. In this way, each forecast is computed only from the information that would have been available at the time of the forecast. We set up such a test set in Chapter 11.

You can run this chapter's notebook in two ways. In the cloud, [**open ch-11-forecasting.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-11-forecasting.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; only a free Google account is required for Colab.


In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly statsmodels
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

| Tool | Why we use it here | Alternatives | Trade-off |
|---|---|---|---|
| statsmodels, time-series module (`statsmodels.tsa`) | Classical decomposition and exponential smoothing, fitted with the same `.fit()` call as the regressions of Chapter 8 | Prophet (from Meta), statsforecast (from Nixtla), sktime | statsmodels includes the classical methods of this chapter; the alternatives also include automatic model selection and more model families |

## From orders to a weekly series
In the previous chapters, we worked with tables in which each row contained the values of many variables. In the customer snapshot, used from [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) to [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html), each customer is described at one point in time, a fixed cutoff date. The order table of [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) contains orders from many different dates, with one row per order line, that is, one product in one order. A **time series**, in contrast, is one variable measured at regular intervals, such as revenue per week. Its values are in time order, and the order itself is informative: the value of one week is related to the values of earlier weeks. We have already computed one time series. In [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html), the monthly revenue for a line chart was computed from the order lines with this line of code:

```python
monthly = lines.set_index("order_date")["line_total"].resample("ME").sum().reset_index()
```

The result has one row per month, with the total of `line_total` for that month. In Chapter 6, the monthly revenue was used only in a chart. In this chapter, we compute weekly revenue from `line_total` in the same way and forecast the revenue of future weeks.

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])

# --- Look: the first and last order dates, and the first order lines ---
print(orders["order_date"].min(), orders["order_date"].max())
orders[["order_id", "order_date", "line_total"]].head(5)

The first order date is Monday, July 3, 2023, and the last is Friday, June 26, 2026. Orders are placed from Monday to Friday. Each date is printed with the time 00:00:00, midnight, because every date is stored as a timestamp, a date with a time of day. In the first five rows, order O000700 has four lines, all dated July 3, 2023.

To turn order lines into weekly revenue, we add up `line_total` within each calendar week:

In [ ]:
weekly = orders.set_index("order_date")["line_total"].resample("W").sum()
weekly.index.name = "week"            # a shorter label for the dates

print(len(weekly), "weeks")
weekly.head(3)

In the first line, four steps are chained:

| Step | Result |
|---|---|
| `.set_index("order_date")` | The table, with `order_date` moved into the index (the row labels). In [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html), the grouping key was moved into the index as a side effect of `groupby`. Here we move the column on purpose, because dates in the index are required by `resample`. |
| `["line_total"]` | One column, a Series with the dates as its index. |
| `.resample("W")` | The rows grouped by calendar week. `"W"` stands for weeks that end on Sunday, and each week is labeled with its Sunday date. In [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html) we used `"ME"`, for month end. |
| `.sum()` | The total of `line_total` within each week: one number per week. |

`weekly` is a Series with 156 values, one per week. In the line `Freq: W-SUN` below the values, `W-SUN` means weekly rows, with weeks ending on Sunday. The first value, \$123,770.09, is labeled July 9, 2023, the last day of the first week. We check this value against the order lines:

In [ ]:
# --- Look: the order lines of the first week add up to the first weekly value ---
first_week = orders[orders["order_date"] <= "2023-07-09"]
print(len(first_week), "order lines")
first_week["line_total"].sum().round(2)

The 867 order lines dated on or before July 9, 2023 add up to the same \$123,770.09. Because the data begins on a Monday and ends on a Friday, the first and the last week each contain a full working week. As a result, none of the 156 weeks is partial.

To draw the series, we turn `weekly` back into a table with `reset_index`, which moves the weeks from the index into a column named `week`, and draw the table with `px.line`, as for the monthly revenue in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html):

In [ ]:
px.line(weekly.reset_index(), x="week", y="line_total",
        labels={"line_total": "Weekly revenue ($)", "week": "Week"})

## Trend, seasonality, and noise

In the figure, three kinds of movement can be seen: a slow increase, yearly peaks, and variation from week to week. These correspond to the three components by which a time series is usually described:

| Component | Meaning | In this series |
|---|---|---|
| **Trend** | the long-run direction of the series | a slow increase |
| **Seasonality** | a pattern that repeats on a fixed calendar, here every year | the August and January school stock-ups of [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) |
| **Noise** | the variation that remains | the rises and falls of individual weeks |

In a **classical decomposition**, a series is separated into these three components. statsmodels, which we used for regression in [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), has a time-series module, `statsmodels.tsa`, with a function for this decomposition, `seasonal_decompose`. With `period=52`, the function is told that the pattern repeats every 52 rows. The three components are computed in order:

1. **Trend.** For each week, the average revenue of the 52 weeks around that week, 26 on each side. The trend is missing for the first and the last 26 weeks of the series, because a full year of weeks around each of them is not available.
2. **Seasonal.** For each week, the gap between revenue and trend. These gaps are averaged by week of the year over the years in the data, and adjusted so that the 52 values average zero over a year. The same 52 values are repeated every year.
3. **Residual.** Revenue minus trend minus seasonal: the noise.

Revenue is therefore the sum of the three components:

$$\text{revenue} = \text{trend} + \text{seasonal} + \text{residual}$$

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(weekly, period=52)

# one column per component, with one row per week
parts = pd.DataFrame({
    "observed": weekly,
    "trend": decomp.trend,
    "seasonal": decomp.seasonal,
    "residual": decomp.resid,
})

# --- Look: the four values of one peak week ---
parts.loc["2024-08-18"].round(0)

`decomp.trend`, `decomp.seasonal`, and `decomp.resid` are Series with one value per week. In `parts`, the three Series are placed side by side with the weekly revenue. With `.loc`, as in [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html), we look up one row by its label. A date label can be written as a string. In the week ending August 18, 2024, revenue was \$196,471. The trend was \$115,638, and the seasonal value for that week of the year was \$65,491. The remaining \$15,343 is the residual. The three components add up to revenue: 115,638 + 65,491 + 15,343 = 196,472, a difference of one dollar from rounding.

- With `.iloc`, rows are selected by their position, counted from 0, like the list indices of [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html). With `.loc`, rows are selected by their label.
- Negative positions are counted from the end, and the stop position of a slice is excluded. `weekly.iloc[:-13]` is every week except the last 13, and `weekly.iloc[-13:]` is the last 13 weeks.
- With a list of positions, in a second pair of brackets, several rows are selected: `.iloc[[0, -1]]` is the first and the last row.

With `.iloc`, we look at the first and last weeks that have a trend value:

In [ ]:
# --- Look: the first and last weeks that have a trend value ---
parts.dropna().iloc[[0, -1]].round(0)

With `dropna`, the weeks without a trend value are removed. Where a trend value exists, the trend increases from \$111,498 a week in early January 2024 to \$120,174 a week at the end of December 2025, about 8% in two years. The six largest seasonal values are from mid-August to early September and in January:

In [ ]:
# the seasonal values repeat every year, so the first 52 weeks hold all of them
parts["seasonal"].iloc[:52].nlargest(6).round(0)

The weeks ending August 13 to September 3 lie \$65,491 to \$89,777 above the trend, and the first two weeks of January \$35,453 and \$38,462 above the trend. The dates in this output are those of the first year. The same values apply to the same weeks of every year.

To draw the components as stacked panels, we reshape `parts` with `melt`, the long-form reshape of [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html). With `facet_row="component"`, each component is drawn in its own panel.

In [ ]:
# with melt, the four columns are stacked into one, next to a column naming the component
long = parts.reset_index().melt(id_vars="week", var_name="component", value_name="dollars")
fig = px.line(long, x="week", y="dollars", facet_row="component", height=620,
              labels={"dollars": "", "week": ""})

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
# labels on the chart: panel names, the rise in the trend, and the two seasonal peaks of one year
_axes = {}
for _t in fig.data:
    _axes[_t.hovertemplate.split("component=")[1].split("<")[0]] = (_t.xaxis, _t.yaxis)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))
fig.update_yaxes(tickprefix="$")

_xr, _yr = _axes["trend"]
_tr = parts["trend"].dropna()
_first, _last = _tr.index[0], _tr.index[-1]
fig.add_shape(type="line", xref=_xr, yref=_yr, x0=str(_first), x1=str(_last), y0=_tr.iloc[0], y1=_tr.iloc[0],
              line=dict(color="#9aa1a9", width=1.5, dash="dot"))
fig.add_annotation(xref=_xr, yref=_yr, x=str(_first), y=_tr.iloc[0], ax=0, ay=-34, showarrow=True, arrowhead=2,
                   text=f"${_tr.iloc[0]:,.0f}, {_first:%b %Y}", font=dict(size=12))
fig.add_annotation(xref=_xr, yref=_yr, x=str(_last), y=_tr.iloc[-1], ax=0, ay=-34, showarrow=True, arrowhead=2,
                   text=f"${_tr.iloc[-1]:,.0f}, {_last:%b %Y} (+{_tr.iloc[-1] / _tr.iloc[0] - 1:.0%})",
                   font=dict(size=12))

_xs, _ys = _axes["seasonal"]
_se = parts["seasonal"]
_aug = _se["2024-08-01":"2024-09-30"]
_jan = _se["2025-01-01":"2025-01-31"]
_aug_top = _aug.nlargest(4)
_jan_top = _jan.nlargest(2)
fig.add_annotation(xref=_xs, yref=_ys, x=str(_aug.idxmax()), y=_aug.max(), ax=-40, ay=-28, showarrow=True,
                   arrowhead=2, xanchor="right", font=dict(size=12),
                   text=f"Mid-August to early September:<br>+&#36;{_aug_top.min() / 1000:,.0f}k to +&#36;{_aug_top.max() / 1000:,.0f}k")
fig.add_annotation(xref=_xs, yref=_ys, x=str(_jan.idxmax()), y=_jan.max(), ax=40, ay=-34, showarrow=True,
                   arrowhead=2, xanchor="left", font=dict(size=12),
                   text=f"First two weeks of January:<br>+&#36;{_jan_top.min() / 1000:,.0f}k to +&#36;{_jan_top.max() / 1000:,.0f}k")
fig

We use the decomposition only to describe the series. The trend of the decomposition at each week is an average that includes the following 26 weeks. At the time of a forecast, those weeks are still unknown, so a forecast computed from the decomposition would contain leakage (Chapter 9). For that reason, no forecast in this chapter is computed from the decomposition.

## A holdout in time order
In [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) we split the customers into a training set and a test set at random. If we split the weeks of a time series at random, the training set contains weeks from after the test weeks. This is leakage, as defined in Chapter 9: information from the period being predicted is available during fitting. For example, a test week in May 2026 could be surrounded by April and June weeks in the training set. May could then be estimated from the April and June weeks on both sides. When a forecast is made in practice, the weeks after the forecast date have not happened yet.

We therefore split the weeks in time order. Every week up to a cutoff date is used for fitting, and the weeks after the cutoff are held out. The held-out weeks are called the **holdout**, the time-series counterpart of the test set. Because the warehouse managers plan one quarter ahead, we hold out the last 13 weeks:

In [ ]:
HOLDOUT = 13                       # weeks held out: one quarter
train = weekly.iloc[:-HOLDOUT]     # every week except the last 13
test = weekly.iloc[-HOLDOUT:]      # the last 13 weeks

print(len(train), "training weeks, the last one ending", train.index[-1])
print(len(test), "holdout weeks, from", test.index[0], "to", test.index[-1])

The 143 training weeks end on March 29, 2026, and the 13 holdout weeks run from the week ending April 5 to the week ending June 28, 2026. Every forecast in the next three sections is computed from `train` alone. `test` is used only to score the forecasts.

In the figure, the two ways of splitting the 156 weeks are compared.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as np
import plotly.graph_objects as go

_fig_rng = np.random.default_rng(0)
_fig_random = np.zeros(len(weekly), dtype=int)
_fig_random[_fig_rng.choice(len(weekly), size=round(0.3 * len(weekly)), replace=False)] = 1
_fig_time = np.zeros(len(weekly), dtype=int)
_fig_time[-HOLDOUT:] = 1                                 # the last 13 weeks, as in `test`
_fig = go.Figure(go.Heatmap(
    z=[_fig_time, _fig_random], x=weekly.index,
    y=["In time order<br>(this chapter)", "At random<br>(as in Chapter 9)"],
    colorscale=[[0, "#9cc3e6"], [1, "#c0392b"]], zmin=0, zmax=1, showscale=False, xgap=1, ygap=10,
    hovertemplate="week ending %{x|%b %d, %Y}<extra></extra>"))
_fig.update_layout(height=220, margin=dict(l=10, r=10, t=10, b=30), plot_bgcolor="rgba(0,0,0,0)")
_fig.update_xaxes(tickformat="%b %Y", dtick="M6")
_fig

## Three baselines

A forecasting method is useful only if it is more accurate than a baseline. As in Chapters 8 to 10, a baseline is a simple method with which a more elaborate method is compared. In [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), the baseline was the mean revenue of all customers, and in [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html), the naive model. A simple forecast can be difficult to improve on. In the rolling backtest at the end of this chapter, for example, the average error of the seasonal naive forecast, which repeats the revenue of the same week one year earlier, is only about \$1,270 a week higher than that of the best method. We therefore compute three baselines before we fit the more elaborate method of the next section. The first two are time-series versions of the naive model of Chapter 9, in which a past value is used as the forecast:

| Baseline | Forecast for each holdout week | Formula |
|---|---|---|
| **Naive** | the revenue of the last training week | $\text{forecast}_{T+h} = \text{revenue}_T$ |
| **Seasonal naive** | the revenue of the same week one year earlier, 52 weeks back | $\text{forecast}_{T+h} = \text{revenue}_{T+h-52}$ |
| **8-week moving average** | the mean revenue of the last 8 training weeks | $\text{forecast}_{T+h} = \frac{1}{8}(\text{revenue}_{T-7} + \dots + \text{revenue}_T)$ |

In the formulas, $T$ is the last training week, and $T+h$ is the holdout week $h$ weeks later. $\text{revenue}_T$ is the revenue of week $T$.

For the seasonal naive forecast we use `shift(52)`. `shift(52)` returns the series with every value moved down by 52 rows. Each week's row then contains the revenue of the week 52 rows earlier:

In [ ]:
# --- Look: with shift(52), every value is moved down by 52 rows ---
check = pd.DataFrame({"revenue": weekly, "52 weeks earlier": weekly.shift(52)})
check.iloc[[0, 52, len(train)]]

Because there is no week a year before the first week, the shifted value of the first week is missing (`NaN`). In the week ending July 7, 2024, row 52, the shifted value is the revenue of the first week, \$123,770.09. The third row is the first holdout week. The seasonal naive forecast for that week is the revenue of the week ending April 6, 2025. For the first four holdout weeks:

| Holdout week ending | Actual revenue | Same week a year earlier | Seasonal naive forecast |
|---|---|---|---|
| April 5, 2026 | \$101,278 | April 6, 2025 | \$124,337 |
| April 12, 2026 | \$122,590 | April 13, 2025 | \$120,362 |
| April 19, 2026 | \$122,370 | April 20, 2025 | \$127,092 |
| April 26, 2026 | \$163,868 | April 27, 2025 | \$106,018 |

Every seasonal naive forecast is taken from a training week. Because the holdout has only 13 weeks, $h$ is at most 13, and the week $T+h-52$ lies before $T$, inside the training period. No holdout value is therefore used.

We now compute the three baseline forecasts for the 13 holdout weeks, one column per method:

In [ ]:
# an empty table with one row per holdout week; each column will hold one method
forecasts = pd.DataFrame(index=test.index)

forecasts["naive"] = train.iloc[-1]                        # one number, repeated
forecasts["seasonal_naive"] = weekly.shift(52).iloc[-HOLDOUT:]
forecasts["moving_avg_8"] = train.iloc[-8:].mean()         # one number, repeated

forecasts.head(4).round(0)

`pd.DataFrame(index=test.index)` returns an empty table with the 13 holdout weeks as its rows. When we assign a single number to a column, every row is given that number. When we assign a Series, each row is given the value for its week. In the naive column, \$120,852 is repeated in every row, and in the moving-average column \$102,761. The values in the seasonal naive column differ from week to week. In the figure, the training weeks from which each baseline is computed are drawn together with its forecast for the holdout.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _subplots

_shown = train["2025-02-01":]
_half = pd.Timedelta(days=3.5)
_hover = "Week ending %{x|%b %d, %Y}<br>$%{y:,.0f}<extra></extra>"
_last8 = train.iloc[-8:]
_src = weekly.iloc[len(train) - 52:len(train) - 52 + HOLDOUT]       # the same weeks one year earlier
_colors = {"naive": "#bf8700", "moving_avg_8": "#0969da", "seasonal_naive": "#cf222e"}
_titles = ["<b>Naive:</b> the revenue of the last training week",
           "<b>8-week moving average:</b> the mean of the last 8 training weeks",
           "<b>Seasonal naive:</b> the same week one year earlier"]
_top = max(_shown.max(), test.max()) * 1.15
_f = _subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.09, subplot_titles=_titles)
for _r, _key in enumerate(["naive", "moving_avg_8", "seasonal_naive"], start=1):
    _c = _colors[_key]
    _ax = "x" if _r == 1 else f"x{_r}"
    _ay = "y" if _r == 1 else f"y{_r}"
    _f.add_shape(type="rect", xref=_ax, yref=f"{_ay} domain", x0=str(test.index[0] - _half),
                 x1=str(test.index[-1] + _half), y0=0, y1=1, fillcolor="rgba(207,34,46,0.08)", line_width=0,
                 layer="below")
    if _r == 1:
        _f.add_annotation(xref=_ax, yref=f"{_ay} domain", x=str(test.index[HOLDOUT // 2]), y=1, yanchor="bottom",
                          showarrow=False, text="Holdout", font=dict(size=12))
    _f.add_trace(_go.Scatter(x=_shown.index, y=_shown, mode="lines", line=dict(color="#b8bec5", width=1.8),
                             name="Training weeks", showlegend=_r == 1, legendgroup="train",
                             hovertemplate=_hover), row=_r, col=1)
    if _key == "naive":
        _f.add_trace(_go.Scatter(x=[str(train.index[-1])], y=[train.iloc[-1]], mode="markers",
                                 marker=dict(size=10, color=_c), showlegend=False, hovertemplate=_hover),
                     row=_r, col=1)
    elif _key == "moving_avg_8":
        _f.add_trace(_go.Scatter(x=_last8.index, y=_last8, mode="lines+markers", line=dict(color=_c, width=2.5),
                                 marker=dict(size=6), showlegend=False, hovertemplate=_hover), row=_r, col=1)
        _f.add_trace(_go.Scatter(x=[str(_last8.index[0]), str(_last8.index[-1])], y=[_last8.mean()] * 2, mode="lines",
                                 line=dict(color=_c, width=1.5, dash="dot"), showlegend=False, hoverinfo="skip"),
                     row=_r, col=1)
    else:
        _f.add_trace(_go.Scatter(x=_src.index, y=_src, mode="lines+markers", line=dict(color=_c, width=2.5),
                                 marker=dict(size=6), showlegend=False, hovertemplate=_hover), row=_r, col=1)
        _f.add_annotation(x=str(forecasts.index[HOLDOUT // 2]), y=_top * 0.9, ax=str(_src.index[HOLDOUT // 2]),
                          ay=_top * 0.9, xref=_ax, yref=_ay, axref=_ax, ayref=_ay, showarrow=True, arrowhead=2,
                          arrowwidth=2, arrowcolor=_c, text="")
        _f.add_annotation(x=str(_src.index[HOLDOUT // 2] + pd.Timedelta(weeks=26)), y=_top * 0.9, xref=_ax,
                          yref=_ay, yshift=11, showarrow=False, text="52 weeks forward",
                          font=dict(color=_c, size=12))
    _f.add_trace(_go.Scatter(x=forecasts.index, y=forecasts[_key], mode="lines+markers",
                             line=dict(color=_c, width=2.5, dash="dash"), marker=dict(size=6, symbol="circle-open"),
                             name="Forecast", showlegend=False, hovertemplate=_hover), row=_r, col=1)
    if _key != "seasonal_naive":
        _px = train.index[-1] if _key == "naive" else _last8.index[3]
        _f.add_annotation(xref=_ax, yref=_ay, x=str(_px), y=forecasts[_key].iloc[-1], ax=-70, ay=-52,
                          xanchor="right", showarrow=True, arrowhead=2, arrowcolor=_c, font=dict(color=_c, size=12),
                          text=f"{'Last training week' if _key == 'naive' else 'Mean of 8 weeks'}: "
                               f"&#36;{forecasts[_key].iloc[-1]:,.0f}")
    _f.add_trace(_go.Scatter(x=test.index, y=test, mode="lines+markers", line=dict(color="#1f2328", width=2),
                             marker=dict(size=5), name="Actual revenue, holdout", showlegend=_r == 1,
                             legendgroup="actual", hovertemplate=_hover), row=_r, col=1)
_f.update_layout(height=700, margin=dict(l=10, r=10, t=30, b=10), hovermode="closest",
                 legend=dict(orientation="h", y=-0.06, x=0))
_f.update_yaxes(tickprefix="$", range=[0, _top])
_f.update_annotations(selector=dict(text=_titles[0]), xanchor="left", x=0)
for _t in _titles[1:]:
    _f.update_annotations(selector=dict(text=_t), xanchor="left", x=0)
_f.update_xaxes(tickformat="%b %Y", dtick="M3")
_f

## Exponential smoothing

Each baseline is based on one week or on a fixed set of weeks. In **exponential smoothing**, the forecast is a weighted average of all past weeks. The weights decrease the further back a week lies. We build the method in two steps: a smoothed level first, then trend and seasonality.

### The smoothed level

In the simplest version, **simple exponential smoothing**, we keep track of one number, the **level**, an estimate of typical weekly revenue. Each week, the level is updated with that week's revenue:

$$\text{new level} = \alpha \times \text{this week's revenue} + (1 - \alpha) \times \text{previous level}$$

The **smoothing weight** $\alpha$ (alpha) is a number between 0 and 1. $\alpha$ is the weight on the newest week, and the previous level is given the remaining weight, $1 - \alpha$. The forecast for every future week is the latest level. We compute the first five levels with $\alpha = 0.2$:

In [ ]:
alpha = 0.2
level = weekly.iloc[0]                 # start the level at the first week's revenue
rows = [[weekly.index[0], weekly.iloc[0], level]]

for i in range(1, 5):                  # weeks 1 to 4 (positions)
    level = alpha * weekly.iloc[i] + (1 - alpha) * level
    rows.append([weekly.index[i], weekly.iloc[i], level])

pd.DataFrame(rows, columns=["week", "revenue", "level"]).set_index("week").round(0)

We start the level at the first week's revenue, \$123,770. In the second week, revenue was \$82,609. The new level is therefore 0.2 × 82,609 + 0.8 × 123,770 = \$115,538. Each week, the level is moved one fifth of the way toward that week's revenue.

Because the update is repeated every week, the weight of an older week is multiplied by $1 - \alpha$ once for each later week:

$$\text{weight of the week } k \text{ weeks back} = \alpha \times (1 - \alpha)^k$$

At $\alpha = 0.2$, for example, the weight of the week two weeks back is $0.2 \times 0.8^2 = 0.128$:

| Weeks back | 0 (newest) | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Weight at $\alpha = 0.2$ | 0.200 | 0.160 | 0.128 | 0.102 | 0.082 |
| Weight at $\alpha = 0.5$ | 0.500 | 0.250 | 0.125 | 0.063 | 0.031 |

The weights decrease by the same factor with every week further back. The method is named after this exponential decrease. With a small $\alpha$, the weight is spread over many past weeks. The level then changes slowly. With a large $\alpha$, most of the weight is on the last few weeks. The level then rises and falls with the week-to-week noise.

The online edition has an interactive explorer here: the smoothed level over the 143 training weeks, with a slider for $\alpha$, bars for the weights of the last twelve weeks, and the average one-week-ahead miss. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/smoothing.html).


### Trend and seasonality

A trend or a seasonal pattern cannot be represented by a single level. In **Holt-Winters exponential smoothing**, two more components are added. Each component is updated in the same way as the level, with its own smoothing weight:

| Component | Estimate of | Smoothing weight |
|---|---|---|
| Level | typical weekly revenue, with the seasonal effect removed | $\alpha$ (alpha) |
| Trend | the change in the level from one week to the next | $\beta$ (beta) |
| Seasonal | 52 values, one per week of the year: how far that week usually lies above or below the level | $\gamma$ (gamma) |

The level and the trend are updated in the same form as the level of the previous section. For the seasonal value, the newest information is the gap between this week's revenue and the level plus trend expected for the week:

$$\text{new seasonal value} = \gamma \times (\text{revenue} - \text{previous level} - \text{trend}) + (1 - \gamma) \times \text{seasonal value 52 weeks earlier}$$

Each of the 52 seasonal values is therefore updated once a year, in its own week of the year.

The forecast for the week $h$ weeks after the last training week is assembled from the final values of the three components:

$$\text{forecast} = \text{level} + h \times \text{trend} + \text{seasonal value for that week of the year}$$

In statsmodels, the method is the class `ExponentialSmoothing`, in the module `statsmodels.tsa.holtwinters`. We label the forecasts of this method `ETS`, the usual abbreviation for exponential smoothing models (error, trend, seasonality).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets = ExponentialSmoothing(train, trend="add", seasonal="add",
                           seasonal_periods=52).fit()

print(f"alpha (level):    {ets.params['smoothing_level']:.3f}")
print(f"beta (trend):     {ets.params['smoothing_trend']:.3f}")
print(f"gamma (seasonal): {ets.params['smoothing_seasonal']:.3f}")

The model is specified by its arguments:

| Argument | Meaning |
|---|---|
| `train` | the series to fit, here only the 143 training weeks |
| `trend="add"` | a trend is included and added to the level, in dollars per week |
| `seasonal="add"` | seasonal values are included and added, in dollars. With `"mul"`, the level would be multiplied by a seasonal factor instead, for a pattern that grows with the level. |
| `seasonal_periods=52` | the seasonal pattern has a length of 52 rows |

The defaults are `trend=None` and `seasonal=None`. Without the two arguments, the model is the smoothed level of the previous section. With `.fit()`, the three smoothing weights and the starting values of the three components are estimated. These values are chosen to make the sum of squared one-week-ahead errors over the training weeks as small as possible, the least-squares criterion of Chapter 8. `ets.params` is a dictionary of the fitted values. We look up the three smoothing weights by their keys.

The fitted weights can be read as follows:

- **$\alpha = 0.084$.** Each week, the level is moved 8.4% of the way toward that week's revenue, net of the seasonal value for that week. From a weight this small, we can see that, once the seasonal pattern is removed, the level of this series changes slowly relative to the week-to-week noise.
- **$\beta = 0$.** The trend is never updated and stays at the starting value, about \$33 a week.
- **$\gamma = 0.116$.** Each week, the seasonal value for that week of the year is moved 11.6% of the way toward the value implied by that week's revenue.

The final values of the components are stored in the fitted model. We assemble the forecast for the first holdout week, $h = 1$, by hand:

In [ ]:
# --- Look: the first holdout week's forecast, assembled from its components ---
last_level = ets.level.iloc[-1]       # level at the last training week
last_trend = ets.trend.iloc[-1]       # trend at the last training week, per week
season_1 = ets.season.iloc[-52]       # seasonal value for this week of the year
print(f"{last_level:,.0f} + 1 × {last_trend:,.0f} + ({season_1:,.0f}) "
      f"= {last_level + last_trend + season_1:,.0f}")

`ets.level`, `ets.trend`, and `ets.season` are Series with one value per training week. The seasonal value for the first holdout week was last updated 52 weeks earlier, at position `-52`. The forecast of \$100,156 is the sum of the level, \$114,599, one week of trend, and a seasonal value of −\$14,476 for early April. `ets.forecast` returns the same sum for each of the 13 holdout weeks:

In [ ]:
forecasts["ETS"] = ets.forecast(HOLDOUT)    # 13 forecasts, labeled by week
forecasts.head(4).round(0)

`ets.forecast(13)` returns a Series of 13 forecasts labeled with their weeks. Because the labels match the rows of `forecasts`, each forecast is placed in the row of its week. The first ETS forecast is the \$100,156 computed above.

The online edition has an interactive explorer here: the 13 holdout weeks with the ETS forecast for each, split into the level and trend and the seasonal value, with the actual revenue revealed week by week. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/forecast-assembly.html).


## The scoreboard

With four forecasts for each holdout week, we can now compare the methods. In Chapters 9 and 10, each prediction was either right or wrong. A forecast is a number, so we measure instead the distance between the forecast and the actual revenue. We score the four methods on the holdout with two metrics:

| Metric | Formula | Unit |
|---|---|---|
| **MAE**, mean absolute error | $\displaystyle \frac{1}{n} \sum_{t=1}^{n} \lvert \text{actual}_t - \text{forecast}_t \rvert$ | dollars per week |
| **MAPE**, mean absolute percentage error | $\displaystyle \frac{100}{n} \sum_{t=1}^{n} \frac{\lvert \text{actual}_t - \text{forecast}_t \rvert}{\text{actual}_t}$ | percent of actual revenue |

In both formulas, $n$ is the number of holdout weeks, 13, and $t$ numbers the holdout weeks from 1 to $n$. The symbol $\sum$ stands for the sum over these weeks, so $\frac{1}{n} \sum$ is the average over the holdout weeks.

With the absolute value, the sign is dropped. A forecast that is too high and one that is too low by the same amount therefore count equally. We first compute both errors for three weeks of the ETS forecast:

In [ ]:
# --- Look: the ETS errors in the first three holdout weeks ---
errors = pd.DataFrame({"actual": test, "forecast": forecasts["ETS"]})
errors["error"] = errors["actual"] - errors["forecast"]
errors["abs error"] = errors["error"].abs()
errors["abs % error"] = errors["abs error"] / errors["actual"] * 100
errors.head(3).round(1)

In the week ending April 19, the ETS forecast was \$111,917, and revenue was \$122,370. The absolute error and the absolute percentage error are:

$$\lvert 122{,}370 - 111{,}917 \rvert = 10{,}453, \qquad \frac{10{,}453}{122{,}370} \times 100 = 8.5\%$$

MAE is the mean of the `abs error` column over all 13 weeks, and MAPE is the mean of the `abs % error` column. We compute both for every method in a loop over the column names of `forecasts`, `forecasts.columns`:

In [ ]:
rows = []
for method in forecasts.columns:
    abs_error = (test - forecasts[method]).abs()      # one value per holdout week
    rows.append([method, abs_error.mean(), (abs_error / test).mean() * 100])

scoreboard = pd.DataFrame(rows, columns=["method", "MAE ($)", "MAPE (%)"])
scoreboard.sort_values("MAE ($)").round(1)

On this holdout, ETS has the lowest MAE, \$17,433 a week, and the lowest MAPE, 14.2%. The naive forecast is second on MAE (\$19,595), and the seasonal naive is last on both metrics (\$23,132 and 21.1%). In the seasonal panel of the figure, April to June has no peak. As a result, the values copied from last year consist mostly of last year's noise.

To draw the forecasts and the actual revenue in one chart, we add the actual revenue as a column of a copy of `forecasts`. With `.copy()`, as in Chapter 9, `forecasts` itself is left unchanged. From a wide table, one line is drawn per column, as in [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html).

In [ ]:
plot_df = forecasts.copy()
plot_df["actual"] = test
px.line(plot_df, markers=True, labels={"value": "Weekly revenue ($)", "variable": "series"},
        color_discrete_map={"naive": "#bf8700", "seasonal_naive": "#cf222e",
                            "moving_avg_8": "#0969da", "ETS": "#8250df", "actual": "#1f2328"})

Second place differs between the two metrics. The naive forecast is second on MAE, and the moving average is second on MAPE (18.2% against 18.5% for the naive forecast). The reason is visible in the percentage errors of the two methods:

In [ ]:
# --- Look: absolute percentage errors of two baselines, week by week ---
pct = pd.DataFrame({
    "actual": test,
    "naive": (test - forecasts["naive"]).abs() / test * 100,
    "moving_avg_8": (test - forecasts["moving_avg_8"]).abs() / test * 100,
})
pct.round(1)

The naive forecast, \$120,852, is far above the three slow weeks at the end of June. The errors of the naive forecast in those weeks are 51.0%, 39.2%, and 44.4% of revenue. The moving average, \$102,761, is far below the two busiest weeks, by \$61,107 and \$56,643. Because those weeks have high revenue, these larger dollar errors are only 37.3% and 35.5% of revenue. In MAPE, therefore, an error in a slow week counts more than the same error in a busy week. The following table lists this limit and the other limits of MAPE:

| Situation | Effect on MAPE |
|---|---|
| A week with zero revenue | The percentage error is a division by zero. MAPE is then undefined, and the result in pandas is `inf`. For a series with weeks of zero revenue, such as one product's sales, we use MAE. |
| Slow weeks and busy weeks | The same dollar error is a larger percentage of a slow week. A \$10,000 error is 12.5% of the \$80,013 week ending June 7 and 6.3% of the \$159,404 week ending May 31. |
| Series of different size | MAE is in dollars, so its size depends on the size of the series. An MAE of \$20,000 a week is moderate for the company as a whole, whose weekly revenue is about \$115,000, but would be very large for a product category with, for example, \$30,000 of revenue a week. To compare the accuracy of forecasts for series of different size, such as the company and one category, two categories, or a large and a small product, we therefore use MAPE, a percentage. |

The scoreboard is based on a single quarter, one sample of forecast accuracy. In the evaluation section we repeat the comparison on the other three quarters of the final year.

With an AI assistant, you can fit a more elaborate model on request and obtain a closer fit to the training weeks. As in Chapter 9, a closer fit may only mean that more of the noise in the training weeks is fitted. The comparison with the baselines on held-out weeks is the part to do yourself. You choose the baselines, the holdout, and the metric, and you check that no forecast was computed from the held-out weeks.

## Prediction intervals
A forecast of \$100,156 for one week is a single number. For staffing and stock, the warehouse managers also need to know how far revenue could plausibly lie from the forecast. A **prediction interval** is a range within which the actual value is expected to fall with a stated probability, such as 95%.

We build the interval from the model's past errors. `ets.fittedvalues` contains, for each training week, the forecast made from the earlier weeks. The **residuals** `train - ets.fittedvalues` are therefore the one-week-ahead errors on the training weeks:

$$\text{residual} = \text{revenue} - \text{fitted value}$$

As in the percentile intervals of [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html), we take the middle 95% of these errors, from the 2.5th to the 97.5th percentile. Then we add both ends to each forecast:

$$\text{forecast} + \text{2.5th percentile of the residuals} \quad \text{to} \quad \text{forecast} + \text{97.5th percentile of the residuals}$$

In [ ]:
resid = train - ets.fittedvalues            # one-week-ahead errors, training weeks
low = resid.quantile(0.025)                 # 2.5% of the errors lie below this
high = resid.quantile(0.975)                # 2.5% of the errors lie above this
print(f"middle 95% of the one-week-ahead errors: {low:,.0f} to {high:,.0f}")

interval = pd.DataFrame({
    "low": forecasts["ETS"] + low,
    "forecast": forecasts["ETS"],
    "high": forecasts["ETS"] + high,
    "actual": test,
})
interval.head(4).round(0)

`quantile(0.025)` returns the value below which 2.5% of a Series lies, as with `np.percentile(..., 2.5)` in Chapter 7. The middle 95% of the one-week-ahead errors runs from −\$37,449 to +\$43,645. For the first holdout week, with a forecast of \$100,156, the range is therefore:

$$100{,}156 - 37{,}449 = 62{,}707 \quad \text{to} \quad 100{,}156 + 43{,}645 = 143{,}801$$

The upper end is farther from the forecast than the lower end. We look at the largest errors in each direction:

In [ ]:
# --- Look: the largest one-week-ahead errors above and below the forecast ---
print(f"largest above: {resid.max():,.0f}, in the week ending {resid.idxmax()}")
print(f"largest below: {resid.min():,.0f}, in the week ending {resid.idxmin()}")

`idxmax`, from [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), returns the label of the largest value, and `idxmin` the label of the smallest. The largest error above the forecast, \$81,207, is the week ending August 10, 2025, the week of the school-district order. The largest error below the forecast, \$53,770, is the week ending January 4, 2026. With a single large order, one week can lie far above the forecast. No training week lies as far below the forecast. We now count the holdout weeks that fall inside their ranges:

In [ ]:
inside = (test >= interval["low"]) & (test <= interval["high"])
print(f"{inside.sum()} of {len(test)} holdout weeks fall inside the range")
interval[~inside].round(0)

Eleven of the 13 holdout weeks lie inside their ranges. The two outside are the spikes of the weeks ending April 26 and May 31. With only 13 weeks, a count of 11 is consistent with a 95% range. The count is too small for us to distinguish a 95% range from a somewhat narrower one. The interval also has one known weakness. The interval is built from one-week-ahead errors, while the holdout forecasts reach up to 13 weeks ahead. Errors can grow with the **horizon**, the number of weeks ahead. The range may therefore be too narrow for the later weeks of the quarter.

The warehouse managers use both ends of the range. Staffing is set near the forecast, and capacity is reserved up to the high end. As with the confidence intervals of Chapter 7, the range is reported together with the single number, because the managers plan around the width of the range.

## Evaluation with a rolling backtest

|  |  |
|---|---|
| **Metric** | MAE in dollars per week, with MAPE alongside. |
| **Test set** | four 13-week holdouts that together cover the final year, July 2025 to June 2026. Each holdout is forecast only from earlier weeks. |
| **Baselines** | the naive forecast, the seasonal naive forecast, and the 8-week moving average. ETS is useful only if its average error over the year is lower than that of every baseline. |

To obtain four holdouts, we move the holdout back through the final year in steps of 13 weeks. For each holdout, every method is refitted on the weeks before that holdout and scored on the holdout. This procedure is called a **rolling backtest**. A rolling backtest is the time-series counterpart of the cross-validation of [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html), in which the training data were divided into folds and each fold was held out in turn. Each holdout corresponds to a fold. In contrast to the folds of Chapter 10, every holdout lies later in time than all of its training weeks.

The figure shows the four holdouts.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
_fig_starts = [104, 117, 130, 143]                       # the holdout start positions in the code below
_fig_rows = []
for _fig_end in _fig_starts:
    _fig_row = np.full(len(weekly), 2)                   # 2: not used
    _fig_row[:_fig_end] = 0                              # 0: training weeks
    _fig_row[_fig_end:_fig_end + HOLDOUT] = 1            # 1: holdout weeks
    _fig_rows.append(_fig_row)
_fig = go.Figure(go.Heatmap(
    z=_fig_rows[::-1], x=weekly.index,
    y=[f"holdout from {weekly.index[s]:%b} {weekly.index[s].day}, {weekly.index[s].year}" for s in _fig_starts][::-1],
    colorscale=[[0, "#9cc3e6"], [0.5, "#c0392b"], [1, "#e1e4e8"]], zmin=0, zmax=2,
    showscale=False, xgap=1, ygap=6, hovertemplate="week ending %{x|%b %d, %Y}<extra></extra>"))
_fig.update_layout(height=260, margin=dict(l=10, r=10, t=10, b=30), plot_bgcolor="rgba(0,0,0,0)")
_fig.update_xaxes(tickformat="%b %Y", dtick="M6")
_fig

To repeat the steps of the previous sections for any holdout, we put these steps in a function. In `forecast_all(series, end)`, the four methods are fitted on the weeks before position `end` of the series. The function returns two objects: the actual revenue of the 13 weeks from position `end` on, and a table of the four forecasts for those weeks.

In [ ]:
def forecast_all(series, end):
    train = series.iloc[:end]                  # the weeks before the holdout
    test = series.iloc[end:end + HOLDOUT]      # the 13 holdout weeks
    f = pd.DataFrame(index=test.index)
    f["naive"] = train.iloc[-1]
    f["seasonal_naive"] = series.shift(52).iloc[end:end + HOLDOUT]
    f["moving_avg_8"] = train.iloc[-8:].mean()
    model = ExponentialSmoothing(train, trend="add", seasonal="add",
                                 seasonal_periods=52).fit()
    f["ETS"] = model.forecast(HOLDOUT)
    return test, f

# the four holdouts start at positions 104, 117, 130, and 143
rows = []
for end in [104, 117, 130, 143]:
    test_w, f = forecast_all(weekly, end)     # two objects, assigned to two names
    for method in f.columns:
        abs_error = (test_w - f[method]).abs()
        rows.append([test_w.index[0], method, abs_error.mean(),
                     (abs_error / test_w).mean() * 100])

backtest = pd.DataFrame(rows, columns=["holdout starts", "method", "MAE ($)", "MAPE (%)"])
print(backtest.shape)
backtest.pivot_table(index="holdout starts", columns="method", values="MAE ($)").round(0)

The code above has four parts:

- **Inside `forecast_all`**, `train` and `test` are **local** names (names that exist only while the function runs). Their values are the slices of one holdout, and the `train` and `test` defined earlier are not changed.
- **The four holdouts** start at positions 104, 117, 130, and 143. Position 104 is the week ending July 6, 2025, after two full years of training weeks. Each later holdout starts 13 weeks later, and the last one, at position 143, is the holdout of the previous sections.
- **`test_w, f = forecast_all(weekly, end)`**: as with `train_test_split` in Chapter 9, the two objects returned by the function are assigned to two names at once.
- **The two loops**: the outer loop is run once per holdout and the inner loop once per method. `backtest` therefore has one row per holdout and method, 16 in all. With `pivot_table` from Chapter 3, we arrange the MAE values in `backtest` as a grid, with one row per holdout and one column per method.

A different method is the most accurate in different quarters. The seasonal naive forecast is best in the quarters starting in July and October 2025, by \$96 a week over ETS in the first. The moving average is best in the quarter starting in January 2026, and ETS in the quarter starting in April. In the quarter that contains the August peak, the errors of the naive forecast and the moving average are about \$38,000 a week. The moving average is also the least accurate method in the following quarter (\$55,497), because the last eight training weeks of that quarter include the four peak weeks of August 2025.

In [ ]:
# average of each method over the four holdouts
backtest.groupby("method")[["MAE ($)", "MAPE (%)"]].mean().sort_values("MAE ($)").round(1)

Averaged over the four quarters, ETS has the lowest MAE, \$22,650 a week, against \$23,921 for the seasonal naive forecast, \$25,388 for the naive forecast, and \$33,589 for the moving average. ETS also has the lowest average MAPE, 19.3%. ETS therefore meets the standard we set. The advantage of ETS over the seasonal naive forecast is about \$1,270 a week, or 5%. This advantage is smaller than the differences between quarters for any single method. On the single-quarter scoreboard, the seasonal naive forecast came last. Averaged across the year, the seasonal naive forecast is second on MAE.

The online edition has a rolling-backtest explorer here: the 13-week holdout is moved through the final year one week at a time, and the four methods are refitted and scored on every holdout. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/rolling-backtest.html).


## The decision this informs

The warehouse managers receive the ETS forecast for the next 13 weeks with its 95% range, refitted each week as new revenue is recorded. Over the final year, the ETS forecast had an MAE of \$22,650 a week and a MAPE of 19.3%. The seasonal naive forecast had an MAE of \$23,921. Because the two are close, the seasonal naive forecast is reported next to the ETS forecast as a check. A week in which the two differ widely is reviewed by hand. The ordering calendar is set from the seasonal values. The weeks from the second week of August to the first week of September lie \$65,000 to \$90,000 above the trend, and the first two weeks of January about \$35,000 to \$38,000 above it. Stock and temporary staff are therefore scheduled ahead of both peaks. Around the August peak, the naive forecast and the moving average are not used at all.

The figure shows both parts of the decision. In the upper panel, ETS is refitted on all 156 weeks, and the forecast for the 13 weeks from July to September 2026 is drawn with its 95% range and the seasonal naive check. In the lower panel, the seasonal values of the decomposition are drawn for one year, with the two peaks used for the ordering calendar highlighted.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing as _ES

# the forecast the managers receive: ETS refitted on all weeks, 13 weeks ahead
_full = _ES(weekly, trend="add", seasonal="add", seasonal_periods=52).fit()
_fc = _full.forecast(HOLDOUT)
_res = weekly - _full.fittedvalues
_lo, _hi = _fc + _res.quantile(0.025), _fc + _res.quantile(0.975)
_sn = weekly.iloc[-52:-52 + HOLDOUT]                              # the same weeks one year earlier
_sn.index = _fc.index
_naive, _ma = weekly.iloc[-1], weekly.iloc[-8:].mean()
_recent = weekly.iloc[-26:]
_hover = "Week ending %{x|%b %d, %Y}<br>$%{y:,.0f}<extra></extra>"

# the ordering calendar: one year of seasonal values from the decomposition, July 2024 to June 2025
_year = decomp.seasonal["2024-07-01":"2025-06-30"]
_aug = _year["2024-08-01":"2024-09-30"].nlargest(4)
_jan = _year["2025-01-01":"2025-01-31"].nlargest(2)
_bar_c = ["#bf8700" if (_d in _aug.index or _d in _jan.index) else "#b8bec5" for _d in _year.index]

_titles = ["<b>Forecast for the next 13 weeks:</b> ETS with its 95% range and the seasonal naive check",
           "<b>Ordering calendar:</b> seasonal value for each week of the year"]
_f = _subplots(rows=2, cols=1, vertical_spacing=0.2, subplot_titles=_titles, row_heights=[0.58, 0.42])
_f.add_trace(_go.Scatter(x=_recent.index, y=_recent, mode="lines", line=dict(color="#b8bec5", width=1.8),
                         name="Weekly revenue", hovertemplate=_hover), row=1, col=1)
_f.add_trace(_go.Scatter(x=[str(_d) for _d in list(_fc.index) + list(_fc.index[::-1])], y=list(_hi) + list(_lo[::-1]), fill="toself",
                         fillcolor="rgba(130,80,223,0.14)", line_width=0, name="95% range", hoverinfo="skip"),
             row=1, col=1)
_f.add_trace(_go.Scatter(x=_fc.index, y=_fc, mode="lines+markers", line=dict(color="#8250df", width=2.5),
                         marker=dict(size=6), name="ETS forecast", hovertemplate=_hover), row=1, col=1)
_f.add_trace(_go.Scatter(x=_sn.index, y=_sn, mode="lines+markers", line=dict(color="#cf222e", width=1.8, dash="dash"),
                         marker=dict(size=5, symbol="circle-open"), name="Seasonal naive (check)",
                         hovertemplate=_hover), row=1, col=1)
_f.add_trace(_go.Scatter(x=[str(_fc.index[0]), str(_fc.index[-1])], y=[_naive, _naive], mode="lines",
                         line=dict(color="#8c959f", width=1.4, dash="dot"), showlegend=False, hoverinfo="skip"),
             row=1, col=1)
_f.add_trace(_go.Scatter(x=[str(_fc.index[0]), str(_fc.index[-1])], y=[_ma, _ma], mode="lines",
                         line=dict(color="#8c959f", width=1.4, dash="dot"), showlegend=False, hoverinfo="skip"),
             row=1, col=1)
_f.add_annotation(x=str(_fc.index[2]), y=min(_naive, _ma), ax=30, ay=95, xanchor="left", showarrow=True, arrowhead=2,
                  arrowcolor="#8c959f", row=1, col=1,
                  text=f"Naive (&#36;{_naive:,.0f}) and 8-week<br>moving average (&#36;{_ma:,.0f}): not used",
                  font=dict(size=11, color="#6e7781"))
_pk = _fc.nlargest(4).sort_index()                                # the four peak weeks of the forecast
_f.add_vrect(x0=str(_pk.index[0] - pd.Timedelta(days=3.5)), x1=str(_pk.index[-1] + pd.Timedelta(days=3.5)),
             fillcolor="rgba(191,135,0,0.12)", line_width=0, layer="below", row=1, col=1)
_f.add_annotation(x=str(_pk.index[0] - pd.Timedelta(days=3.5)), y=0.97, yref="y domain", xanchor="right",
                  yanchor="top", showarrow=False, align="right", xshift=-6,
                  text="August peak:<br>stock and staff<br>scheduled ahead", font=dict(size=12, color="#9a6700"),
                  row=1, col=1)
_f.add_vline(x=str(weekly.index[-1] + pd.Timedelta(days=3.5)), line=dict(color="#8c959f", width=1, dash="dot"),
             row=1, col=1)
_f.add_annotation(x=str(weekly.index[-1]), y=0.02, yref="y domain", xanchor="right", showarrow=False,
                  text="Today", font=dict(size=11, color="#8c959f"), row=1, col=1)

_f.add_trace(_go.Bar(x=_year.index, y=_year, marker_color=_bar_c, showlegend=False,
                     hovertemplate="Week ending %{x|%b %d}<br>%{y:$,.0f}<extra></extra>"), row=2, col=1)
_f.add_annotation(x=str(_aug.idxmax()), y=_aug.max(), xref="x2", yref="y2", ax=50, ay=-6, xanchor="left",
                  showarrow=True, arrowhead=2, font=dict(size=12, color="#9a6700"),
                  text=f"Mid-August to early September:<br>+&#36;{_aug.min() / 1000:,.0f}k to +&#36;{_aug.max() / 1000:,.0f}k")
_f.add_annotation(x=str(_jan.idxmax()), y=_jan.max(), xref="x2", yref="y2", ax=40, ay=-34, xanchor="left",
                  showarrow=True, arrowhead=2, font=dict(size=12, color="#9a6700"),
                  text=f"First two weeks of January:<br>+&#36;{_jan.min() / 1000:,.0f}k to +&#36;{_jan.max() / 1000:,.0f}k")
_f.update_layout(height=680, margin=dict(l=10, r=10, t=40, b=10), hovermode="closest", bargap=0.15,
                 legend=dict(orientation="h", y=0.505, x=0, yanchor="top"))
_f.update_yaxes(tickprefix="$", row=1, col=1)
_f.update_yaxes(tickprefix="$", row=2, col=1)
_f.update_xaxes(tickformat="%b %Y", dtick="M1", row=1, col=1)
_f.update_xaxes(tickformat="%b", dtick="M1", title_text="Week of the year", row=2, col=1)
_f.update_annotations(selector=dict(text=_titles[0]), xanchor="left", x=0)
_f.update_annotations(selector=dict(text=_titles[1]), xanchor="left", x=0)
_f

The forecast is an extension of the patterns in three years of data. Events absent from those data, such as a new large customer, the loss of a school district, or a price change, are not reflected in the forecast. For the loss of a customer, the churn list of [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) is an early warning.

## Exercises


### Build lab

Build the weekly revenue series for the food service product category. Merge the order lines with the `category` column of `pw_products.csv`, as in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html), keep the food service lines, and resample them to weeks as in this chapter. Decompose the series, and compare its six largest seasonal values with the company's. Then compute the four forecasts for the last 13 weeks and report the scoreboard (MAE and MAPE). State in two sentences the week and size of the largest seasonal value in each series, as a percentage of the series' median week, and the method with the lowest MAE and the lowest MAPE on this holdout.

### Evaluate lab

Run the chapter's rolling backtest on the food service series, with the same four holdouts. Report the MAE of each method in each quarter, and each method's average MAE and MAPE over the four quarters. State in two sentences which method has the lowest average MAE and which the lowest average MAPE, and whether ETS meets the chapter's standard for this category: the lowest average MAE of all four methods.

A starter notebook for this lab is provided. In it, the task is restated, the objects from this chapter that are needed for the lab are reproduced, and the cells you complete are marked. [**Open ch-11-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-11-lab-starter.ipynb), or download it from the course page in Blackboard. In Colab, a notebook from GitHub is opened read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.
